## 0) Reproducibility & Environment


This notebook provides a complete, self-contained single-cell RNA-seq workflow.
It downloads data, performs quantification with alevin-fry, conducts clustering with Scanpy, and applies CellTypist for cell-type annotation.
Every step is automated to run reproducibly in CI and to output all artifacts for evaluation.


The environment is initialized to ensure reproducibility by fixing random seeds and controlling thread counts.
Directory structures for runtime data and figures are created, ensuring consistent output paths across local and CI environments.

In [1]:

import os, random, numpy as np, warnings
warnings.filterwarnings("ignore")

random.seed(42); np.random.seed(42)
os.environ["NUMBA_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

BASE = "week6_runtime"
FIGS = "figures"
os.makedirs(BASE, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

print("PWD:", os.getcwd())
print("Dirs:", BASE, FIGS)


PWD: /mnt/e/bioinformatic/week6
Dirs: week6_runtime figures


## 1) Parameters — dataset & whitelist links

This cell defines configuration parameters and shared dataset links.
It stores the Box folder URL for the raw data and an optional whitelist URL, both of which are used to fetch input files automatically if missing.

In [6]:

# Box shared folder (from the assignment). We try multiple download patterns.
BOX_URL = "https://app.box.com/s/lx2xownlrhz3us8496tyu9c4dgade814"

# If you have a direct file link, you can put it here; otherwise we attempt folder-zip patterns.
# Known Box trick: add '?download=1' or use "index.php?rm=box_download_shared_file&shared_name=<id>"
BOX_TRY_URLS = [
    BOX_URL,
    BOX_URL + "?download=1",
    BOX_URL.replace("app.box.com/s/", "app.box.com/index.php?rm=box_download_shared_file&shared_name=")
]

# Whitelist URL (if provided separately). If empty, we'll try to find it inside the Box archive,
# and if that fails, we will synthesize a minimal whitelist in fallback mode.
WHITELIST_URL = "https://raw.githubusercontent.com/f0t1h/3M-february-2018/refs/heads/master/3M-february-2018.txt.gz"  # Put a direct link if you have one; leave empty otherwise.
print("BOX root:", BOX_URL)
print("Whitelist URL:", WHITELIST_URL or "<empty — will try Box/fallback>")


BOX root: https://app.box.com/s/lx2xownlrhz3us8496tyu9c4dgade814
Whitelist URL: https://raw.githubusercontent.com/f0t1h/3M-february-2018/refs/heads/master/3M-february-2018.txt.gz


## 2) Download dataset & whitelist (idempotent)

The script attempts to download the input dataset and whitelist from the Box repository using multiple fallback URLs.
It validates downloads, extracts the archive if present, and lists contents to confirm successful retrieval before analysis.

In [7]:

%%bash
set -euo pipefail

DATA_DIR="week6_runtime/data"
mkdir -p "$DATA_DIR"
cd "$DATA_DIR"

# Try to download Box folder as a zip (best effort). If it fails, we'll fallback later.
if [ ! -f "box_dataset.zip" ]; then
  echo "==> Attempting to fetch Box folder as zip ..."
  urls=( \
    "https://app.box.com/s/lx2xownlrhz3us8496tyu9c4dgade814?download=1" \
    "https://app.box.com/index.php?rm=box_download_shared_file&shared_name=lx2xownlrhz3us8496tyu9c4dgade814" \
  )
  for u in "${urls[@]}"; do
    echo "Trying: $u"
    if curl -fL --retry 3 -o box_dataset.zip "$u"; then
      echo "Downloaded box_dataset.zip"
      break
    fi
    rm -f box_dataset.zip || true
  done
else
  echo "Found box_dataset.zip (skipping download)"
fi

# If we have a zip, try to extract and locate FASTQs, reference and GTF
if [ -f "box_dataset.zip" ] && [ ! -d "box_extracted" ]; then
  echo "==> Extracting box_dataset.zip ..."
  mkdir -p box_extracted
  unzip -qq -o box_dataset.zip -d box_extracted || true
fi

echo "Contents of data dir:"
ls -lah


Found box_dataset.zip (skipping download)
Contents of data dir:


total 24K
drwxrwxrwx 1 pouria pouria 4.0K Nov 12 07:19 .
drwxrwxrwx 1 pouria pouria 4.0K Nov 11 05:59 ..
-rwxrwxrwx 1 pouria pouria    0 Nov 11 05:19 box_data.tar.gz
-rwxrwxrwx 1 pouria pouria    0 Nov 11 05:00 box_data.zip
-rwxrwxrwx 1 pouria pouria  21K Nov 12 07:19 box_dataset.zip
drwxrwxrwx 1 pouria pouria 4.0K Nov 11 04:38 box_extracted
-rwxrwxrwx 1 pouria pouria    0 Nov 11 05:52 toy_read_ref_set.dl


## 3) Locate FASTQs, reference genome (chr5), GTF, and whitelist (or prepare fallback)


This section programmatically locates essential input files — FASTQs, reference genome, GTF, and whitelist — within the extracted dataset.
It builds a structured JSON summary of paths and flags indicating which inputs are available for the quantification stage.

In [22]:
# %% [markdown]
# ### Cell X — Resolve inputs & discover whitelist (handles 3M-february-2018 inside zips)
# We scan `week6_runtime/data/box_extracted/**` for FASTQs, reference, GTF, and a whitelist.
# For whitelist we prefer files named like: 3M-february-2018, whitelist*, 737K* (txt/gz or inside .zip).
# We then write absolute paths to `week6_runtime/out/af_quant/resolved_paths.json`.

import json, os, glob, zipfile, shutil, re, pathlib

BASE = "week6_runtime"
DATA = os.path.join(BASE, "data")
OUT  = os.path.join(BASE, "out", "af_quant")
os.makedirs(OUT, exist_ok=True)

# helper: best-first match finder
def pick_first(patterns):
    for pat in patterns:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    return ""

# locate reference fasta, gtf (you likely already have these)
ref_fa = pick_first([
    os.path.join(DATA, "box_extracted", "**", "*.fa"),
    os.path.join(DATA, "box_extracted", "**", "*.fasta"),
])
ref_gtf = pick_first([
    os.path.join(DATA, "box_extracted", "**", "*.gtf"),
])

# locate FASTQs (toy dataset has selected_R1/selected_R2)
fastq_r1 = pick_first([
    os.path.join(DATA, "box_extracted", "**", "*R1*.fastq"),
    os.path.join(DATA, "box_extracted", "**", "*R1*.fastq.gz"),
])
fastq_r2 = pick_first([
    os.path.join(DATA, "box_extracted", "**", "*R2*.fastq"),
    os.path.join(DATA, "box_extracted", "**", "*R2*.fastq.gz"),
])

# --- Discover a whitelist: prefer 3M-february-2018; also accept names with "white_liste", "whitelist", "737K"
# search both plain files and zip containers
candidates = []
for pat in [
    os.path.join(DATA, "box_extracted", "**", "*3M-february-2018*"),
    os.path.join(DATA, "box_extracted", "**", "*white_liste*"),
    os.path.join(DATA, "box_extracted", "**", "*whitelist*"),
    os.path.join(DATA, "box_extracted", "**", "*737K*"),
]:
    candidates += sorted(glob.glob(pat, recursive=True))

wl_final = ""  # final path to a usable text/gz file containing 16bp barcodes

# If a zip is found, extract a likely txt inside
EXTRACT_DIR = os.path.join(OUT, "wl_zip_extract")
os.makedirs(EXTRACT_DIR, exist_ok=True)
for c in candidates:
    if zipfile.is_zipfile(c):
        try:
            with zipfile.ZipFile(c) as z:
                z.extractall(EXTRACT_DIR)
        except Exception:
            pass

# after extraction, extend candidates with extracted files too
candidates += sorted(glob.glob(os.path.join(EXTRACT_DIR, "**", "*"), recursive=True))

# choose best candidate file name (plain or gz)
def looks_like_wl(path):
    base = os.path.basename(path).lower()
    # accept files with those names, possibly without extension
    if any(tag in base for tag in ["3M-february-2018", "whitelist", "white_liste", "737k"]):
        # ignore obvious non-files (dirs)
        return os.path.isfile(path)
    return False

# rank: prefer 3M-february-2018*, then whitelist*, then 737K*
def rank_key(p):
    b = os.path.basename(p).lower()
    if "3M-february-2018" in b: return (0, len(b))
    if "whitelist" in b or "white_liste" in b: return (1, len(b))
    if "737k" in b: return (2, len(b))
    return (9, len(b))

shortlist = [p for p in candidates if looks_like_wl(p)]
shortlist = sorted(shortlist, key=rank_key)

if shortlist:
    wl_final = os.path.abspath(shortlist[0])

# write flags + resolved paths
flags = {
    "HAVE_FASTQ": bool(fastq_r1 and fastq_r2),
    "HAVE_REF": bool(ref_fa and ref_gtf),
    "HAVE_WL": bool(wl_final)
}
with open(os.path.join(OUT, "status_flags.json"), "w") as f:
    json.dump(flags, f)

resolved = {
    "ref_fa": os.path.abspath(ref_fa) if ref_fa else "",
    "ref_gtf": os.path.abspath(ref_gtf) if ref_gtf else "",
    "fastq_r1": os.path.abspath(fastq_r1) if fastq_r1 else "",
    "fastq_r2": os.path.abspath(fastq_r2) if fastq_r2 else "",
    "whitelist": wl_final,
}
with open(os.path.join(OUT, "resolved_paths.json"), "w") as f:
    json.dump(resolved, f, indent=2)

print("Resolved:")
print(json.dumps(resolved, indent=2))
print("Flags:", flags)


Resolved:
{
  "ref_fa": "/mnt/e/bioinformatic/week6/week6_runtime/data/box_extracted/toy_ref_read/toy_human_ref/fasta/genome.fa",
  "ref_gtf": "/mnt/e/bioinformatic/week6/week6_runtime/data/box_extracted/toy_ref_read/toy_human_ref/genes/genes.gtf",
  "fastq_r1": "/mnt/e/bioinformatic/week6/week6_runtime/data/box_extracted/toy_ref_read/toy_read_fastq/selected_R1_reads.fastq",
  "fastq_r2": "/mnt/e/bioinformatic/week6/week6_runtime/data/box_extracted/toy_ref_read/toy_read_fastq/selected_R2_reads.fastq",
  "whitelist": ""
}
Flags: {'HAVE_FASTQ': True, 'HAVE_REF': True, 'HAVE_WL': False}


In [23]:
%%bash
set -euo pipefail

DATA_DIR="week6_runtime/data"
OUT_DIR="week6_runtime/out/af_quant"
mkdir -p "$DATA_DIR" "$OUT_DIR"

# منبع پایدار (mirror) برای 3M-february-2018
URL="https://zenodo.org/record/3457880/files/3M-february-2018.txt.gz"
DST_GZ="$DATA_DIR/3M-february-2018.txt.gz"
DST_TXT="$DATA_DIR/3M-february-2018"

echo "==> Downloading whitelist from Zenodo mirror ..."
curl -fL --retry 3 -o "$DST_GZ" "$URL"

echo "==> Decompressing ..."
gunzip -c "$DST_GZ" > "$DST_TXT"

# پر کردن resolved_paths.json با این whitelist
python - <<'PY'
import json, os
rp = "week6_runtime/out/af_quant/resolved_paths.json"
with open(rp) as f: obj = json.load(f)
obj["whitelist"] = os.path.abspath("week6_runtime/data/3M-february-2018")
with open(rp, "w") as f: json.dump(obj, f, indent=2)
print("Updated resolved_paths.json with whitelist =", obj["whitelist"])
PY

# چک سریع
head -n 5 "$DST_TXT" || true
echo "✓ ready."


==> Downloading whitelist from Zenodo mirror ...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   279  100   279    0     0    529      0 --:--:-- --:--:-- --:--:--   529
100 17.5M  100 17.5M    0     0  1974k      0  0:00:09  0:00:09 --:--:-- 2106k


==> Decompressing ...
Updated resolved_paths.json with whitelist = /mnt/e/bioinformatic/week6/week6_runtime/data/3M-february-2018
AAACCCAAGAAACACT
AAACCCAAGAAACCAT
AAACCCAAGAAACCCA
AAACCCAAGAAACCCG
AAACCCAAGAAACCTG
✓ ready.


## 4) Quantification — alevin‑fry (with whitelist); **fallback** if inputs missing


If all necessary inputs are found, alevin-fry is executed with the provided whitelist to quantify gene expression.
When real data are unavailable (e.g., CI environment), a fallback synthetic AnnData object is generated to allow the rest of the pipeline to complete deterministically.

In [24]:
%%bash
set -euo pipefail


OUT_DIR="week6_runtime/out/af_quant"
RES_JSON="$OUT_DIR/resolved_paths.json"

# --- Read resolved paths from JSON (set by previous cell) ---
REF_FA=$(python - <<'PY'
import json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json"))["ref_fa"])
PY
)
REF_GTF=$(python - <<'PY'
import json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json"))["ref_gtf"])
PY
)
FASTQ_R1=$(python - <<'PY'
import json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json"))["fastq_r1"])
PY
)
FASTQ_R2=$(python - <<'PY'
import json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json"))["fastq_r2"])
PY
)
# Optional path provided earlier; may be empty
WL_IN=$(python - <<'PY'
import json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json")).get("whitelist",""))
PY
)

# --- If key inputs missing, exit (fallback AnnData already created earlier) ---
if [ -z "${REF_FA}" ] || [ -z "${REF_GTF}" ] || [ -z "${FASTQ_R1}" ] || [ -z "${FASTQ_R2}" ]; then
  echo "⚠️ Missing FASTQ/REF. Using fallback (created earlier)."
  exit 0
fi

# --- Build index with simpleaf ---
IDX_ROOT="$OUT_DIR/refbuild"
mkdir -p "$IDX_ROOT"
echo "==> simpleaf index ..."
simpleaf index \
  -o "$IDX_ROOT" \
  -f "$REF_FA" \
  -g "$REF_GTF" \
  --overwrite

# Locate t2g (name may vary)
T2G=""
for c in "t2g.tsv" "t2g_3col.tsv" "t2g_names.tsv"; do
  if [ -f "$IDX_ROOT/index/$c" ]; then T2G="$IDX_ROOT/index/$c"; break; fi
done
if [ -z "$T2G" ]; then
  echo "❌ Could not find t2g file under $IDX_ROOT/index"
  ls -lah "$IDX_ROOT/index" || true
  exit 1
fi

# ================== SMART WHITELIST HANDLER ==================
CBLEN=16              # 10x v3 barcode length
MIN_VALID=100         # require at least this many unique valid barcodes
USE_WL=0
WL_PATH=""
WL_UTF8="$OUT_DIR/whitelist.utf8.txt"
WL_FILTERED="$OUT_DIR/whitelist.filtered.txt"

# (A) If WL_IN empty or not a file, try to discover common filenames under week6_runtime/data/**
if [ -z "${WL_IN}" ] || [ ! -f "${WL_IN}" ]; then
  echo "🔎 Searching for whitelist under week6_runtime/data ..."
  # Try common names (with or without extension), including the ones you mentioned
  CANDIDATES=$(bash -lc 'shopt -s nullglob globstar; \
    for p in week6_runtime/data/**; do \
      base=$(basename "$p"); \
      case "$base" in \
        *white_liste*|*whitelist*|*3M-february-2018*|*737K* ) echo "$p";; \
      esac; \
    done')
  if [ -n "$CANDIDATES" ]; then
    WL_IN="$(echo "$CANDIDATES" | head -n1)"
    echo "✅ Candidate whitelist: $WL_IN"
  else
    echo "ℹ️ No candidate whitelist found by name."
  fi
else
  echo "✅ Provided whitelist path: $WL_IN"
fi

# (B) Resolve path (support .zip; if gz will be handled later)
if [ -n "${WL_IN}" ] && [ -f "${WL_IN}" ]; then
  MIME=$(file -b --mime-type "${WL_IN}" || echo "")
  if [[ "${WL_IN}" == *.zip ]] || [[ "$MIME" == "application/zip" ]]; then
    echo "==> Extracting whitelist from zip: ${WL_IN}"
    TMP_Z="$OUT_DIR/wl_zip"; rm -rf "$TMP_Z"; mkdir -p "$TMP_Z"
    unzip -qq -o "${WL_IN}" -d "$TMP_Z" || true
    # prefer files that look like relevant names
    if ls "$TMP_Z"/*white_liste*.txt >/dev/null 2>&1; then
      WL_PATH="$(ls -1 "$TMP_Z"/*white_liste*.txt | head -n1)"
    elif ls "$TMP_Z"/*whitelist*.txt >/dev/null 2>&1; then
      WL_PATH="$(ls -1 "$TMP_Z"/*whitelist*.txt | head -n1)"
    elif ls "$TMP_Z"/*3M-february-2018* >/dev/null 2>&1; then
      WL_PATH="$(ls -1 "$TMP_Z"/*3M-february-2018* | head -n1)"
    elif ls "$TMP_Z"/*737K* >/dev/null 2>&1; then
      WL_PATH="$(ls -1 "$TMP_Z"/*737K* | head -n1)"
    elif ls "$TMP_Z"/*.txt >/dev/null 2>&1; then
      WL_PATH="$(ls -1 "$TMP_Z"/*.txt | head -n1)"
    else
      # last resort: any file
      WL_PATH="$(ls -1 "$TMP_Z"/* 2>/dev/null | head -n1 || true)"
    fi
  else
    WL_PATH="${WL_IN}"
  fi
fi

# (C) Convert encoding → UTF-8 (no base sanitizing), strip CR, then VALIDATE exact ACGTN{16}
if [ -n "${WL_PATH}" ] && [ -f "${WL_PATH}" ]; then
  echo "==> Normalizing whitelist to UTF-8: $WL_PATH"
  # if gz, gunzip to temp
  SRC="$WL_PATH"
  if [[ "$WL_PATH" == *.gz ]]; then
    SRC="$OUT_DIR/whitelist.src.txt"
    gunzip -c "$WL_PATH" > "$SRC" || true
  fi
  ENC=$(file -b --mime-encoding "${SRC}" || echo "")
  cp "${SRC}" "${WL_UTF8}"
  if [ "${ENC}" != "utf-8" ] && [ "${ENC}" != "us-ascii" ]; then
    for from in UTF-16LE UTF-16BE UTF-16 UTF-8 WINDOWS-1252; do
      if iconv -f "$from" -t UTF-8 "${SRC}" -o "${WL_UTF8}.try" 2>/dev/null; then
        mv -f "${WL_UTF8}.try" "${WL_UTF8}"
        break
      fi
    done
  fi
  tr -d '\r' < "${WL_UTF8}" > "${WL_UTF8}.nocr" && mv -f "${WL_UTF8}.nocr" "${WL_UTF8}"

  # Keep only exact 16-mer A/C/G/T/N and dedup
  grep -E '^[ACGTN]+$' "${WL_UTF8}" | awk -v L=${CBLEN} 'length($0)==L' | sort -u > "${WL_FILTERED}" || true
  VALID=$(wc -l < "${WL_FILTERED}" || echo 0)
  echo "Whitelist valid lines: ${VALID}"
else
  VALID=0
  echo "ℹ️ No whitelist file to normalize."
fi

# (D) If invalid/empty, try official 10x whitelists; else fall back to --knee
if [ "${VALID}" -ge "${MIN_VALID}" ]; then
  USE_WL=1
  echo "✅ Using explicit whitelist (-x): ${WL_FILTERED}"
  head -n 5 "${WL_FILTERED}" || true
else
  echo "↻ Trying official 10x whitelists ..."
  OFF_DIR="$OUT_DIR/off_wl"; mkdir -p "$OFF_DIR"
  declare -a URLS=(
    "https://raw.githubusercontent.com/10XGenomics/cellranger/master/lib/python/cellranger/barcodes/737K-august-2016.txt.gz"
    "https://raw.githubusercontent.com/10XGenomics/cellranger/master/lib/python/cellranger/barcodes/3M-february-2018.txt.gz"
  )
  for U in "${URLS[@]}"; do
    BN="${OFF_DIR}/$(basename "$U")"
    curl -fL --retry 3 -o "$BN" "$U" || true
    [ -s "$BN" ] || continue
    gunzip -c "$BN" > "${BN%.gz}" || true
    grep -E '^[ACGTN]+$' "${BN%.gz}" | awk -v L=${CBLEN} 'length($0)==L' | sort -u > "${WL_FILTERED}" || true
    VALID=$(wc -l < "${WL_FILTERED}" || echo 0)
    echo "Official whitelist ${U##*/} valid lines: ${VALID}"
    if [ "${VALID}" -ge "${MIN_VALID}" ]; then
      USE_WL=1
      echo "✅ Using official whitelist (-x): ${WL_FILTERED}"
      head -n 5 "${WL_FILTERED}" || true
      break
    fi
  done

  if [ "${USE_WL}" -eq 0 ]; then
    echo "⚠️ No usable whitelist found → will use --knee (auto permit-list)."
  fi
fi
# ================== END WHITELIST HANDLER ==================

# --- Quantification ---
echo "==> simpleaf quant ..."
if [ "${USE_WL}" -eq 1 ]; then
  simpleaf quant \
    -i "$IDX_ROOT/index" \
    -o "$OUT_DIR" \
    -1 "$FASTQ_R1" \
    -2 "$FASTQ_R2" \
    -c 10xv3 \
    -m "$T2G" \
    -x "$WL_FILTERED" \
    -r cr-like \
    --anndata-out \
    -t 4
else
  simpleaf quant \
    -i "$IDX_ROOT/index" \
    -o "$OUT_DIR" \
    -1 "$FASTQ_R1" \
    -2 "$FASTQ_R2" \
    -c 10xv3 \
    -m "$T2G" \
    --knee \
    -r cr-like \
    --anndata-out \
    -t 4
fi

echo "==> Quant finished. Searching for .h5ad ..."
H5AD_FOUND="$(ls -1 "$OUT_DIR"/*.h5ad 2>/dev/null | head -n1 || true)"
if [ -n "$H5AD_FOUND" ]; then
  mv -f "$H5AD_FOUND" "$OUT_DIR/alevin.raw.h5ad"
  echo "Saved: $OUT_DIR/alevin.raw.h5ad"
else
  echo "❌ No .h5ad produced by simpleaf; downstream cell will try MTX→h5ad conversion."
fi

ls -lh "$OUT_DIR" || true


==> simpleaf index ...
2025-11-12T04:51:13.602556Z  INFO simpleaf::simpleaf_commands::indexing: preparing to make reference with roers
2025-11-12T04:51:13.641224Z  INFO grangers::reader::gtf: Finished parsing the input file. Found 3 comments and 2439 records.
2025-11-12T04:51:13.644346Z  INFO roers: Built the Grangers object for 2439 records
2025-11-12T04:51:13.653707Z  WARN grangers::grangers_info: The exon_number column contains null values. Will compute the exon number from exon start position .
2025-11-12T04:51:13.667444Z  INFO roers: Found 2148 exon records from 271 transcripts.
2025-11-12T04:51:18.350053Z  INFO roers: Wrote transcript sequences to output file.
2025-11-12T04:51:18.350103Z  INFO roers: Processing intronic records.
2025-11-12T04:51:18.379040Z  INFO roers: Found 1877 intronic records.
2025-11-12T04:51:18.379192Z  INFO roers: Added flanking length to intronic records.
2025-11-12T04:51:18.384876Z  INFO roers: Merged overlapping intronic records.
2025-11-12T04:51:22.746

CalledProcessError: Command 'b'set -euo pipefail\n\n\nOUT_DIR="week6_runtime/out/af_quant"\nRES_JSON="$OUT_DIR/resolved_paths.json"\n\n# --- Read resolved paths from JSON (set by previous cell) ---\nREF_FA=$(python - <<\'PY\'\nimport json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json"))["ref_fa"])\nPY\n)\nREF_GTF=$(python - <<\'PY\'\nimport json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json"))["ref_gtf"])\nPY\n)\nFASTQ_R1=$(python - <<\'PY\'\nimport json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json"))["fastq_r1"])\nPY\n)\nFASTQ_R2=$(python - <<\'PY\'\nimport json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json"))["fastq_r2"])\nPY\n)\n# Optional path provided earlier; may be empty\nWL_IN=$(python - <<\'PY\'\nimport json; print(json.load(open("week6_runtime/out/af_quant/resolved_paths.json")).get("whitelist",""))\nPY\n)\n\n# --- If key inputs missing, exit (fallback AnnData already created earl

## 5) Convert quant outputs → AnnData (MatrixMarket → `.h5ad`)

Converts raw quantification outputs (MatrixMarket format) into an .h5ad file compatible with Scanpy.
This harmonizes data structure for downstream analysis, regardless of whether it came from real quantification or synthetic fallback.

In [ ]:

import os, glob, anndata as ad, scanpy as sc, scipy.sparse as sp, numpy as np, pandas as pd
from pathlib import Path

OUT_DIR = Path("week6_runtime/out/af_quant")
h5ad_path = OUT_DIR/"alevin.raw.h5ad"

if not h5ad_path.exists():
    # Try to detect matrix market triplet (mtx, genes, barcodes)
    mtx = list(OUT_DIR.rglob("matrix.mtx")) + list(OUT_DIR.rglob("quant.mtx"))
    genes = list(OUT_DIR.rglob("genes.tsv")) + list(OUT_DIR.rglob("features.tsv"))
    barc = list(OUT_DIR.rglob("barcodes.tsv"))
    if mtx and genes and barc:
        import scipy.io
        X = scipy.io.mmread(str(mtx[0])).tocsr()
        var = pd.read_csv(genes[0], sep="\t", header=None)
        if var.shape[1] >= 2:
            var.index = var[1].astype(str).values
        else:
            var.index = [f"GENE{i:06d}" for i in range(X.shape[1])]
        obs = pd.read_csv(barc[0], sep="\t", header=None)
        obs.index = obs[0].astype(str).values
        adata = ad.AnnData(X, obs=obs.iloc[:, :0], var=var.iloc[:, :0])
        adata.write(h5ad_path)
        print("Converted MTX →", h5ad_path)
    else:
        raise FileNotFoundError("No alevin-fry outputs found and no fallback h5ad present.")

adata = sc.read_h5ad(h5ad_path)
print("Loaded AnnData:", adata.shape)


## 6) Scanpy QC → Normalize(Log1p) → HVG → PCA → Neighbors → UMAP → Leiden


Performs core single-cell preprocessing: cell/gene filtering, normalization, log-transformation, highly-variable gene selection, scaling, PCA, neighbor graph construction, UMAP embedding, and Leiden clustering.
The result is a structured low-dimensional representation suitable for visualization and downstream annotation.

In [ ]:

import scanpy as sc, numpy as np

adata.var_names_make_unique()

# Gentle filters for tiny toy data
sc.pp.filter_cells(adata, min_genes=20)
sc.pp.filter_genes(adata, min_cells=2)

# QC metrics
adata.var['mt'] = adata.var_names.str.upper().str.startswith(('MT-','MT_'))
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, inplace=True)

# Normalize/log1p and preserve as raw for CellTypist
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata.copy()

# HVGs (robust caps)
n_hvg = int(min(200, max(adata.n_vars//2, 50)))
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvg)
if (~adata.var["highly_variable"]).all():
    adata.var["highly_variable"] = True
adata = adata[:, adata.var["highly_variable"]].copy()

# Scale + PCA
sc.pp.scale(adata, max_value=10)
n_pcs = int(min(20, max(5, min(adata.n_vars, adata.n_obs)//2)))
sc.tl.pca(adata, n_comps=max(10, n_pcs), svd_solver="arpack")

# Graph + UMAP + Leiden
sc.pp.neighbors(adata, n_neighbors=min(10, max(3, adata.n_obs-1)), n_pcs=n_pcs)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

# Save quick clustering plot
sc.pl.umap(adata, color=["leiden"], save="_clusters.png", show=False)
adata.write("week6_runtime/out/af_quant/adata_processed.h5ad")
print("Saved: figures/umap_clusters.png and adata_processed.h5ad")


## 7) Automatic annotation — CellTypist (with robust fallback for chr5/toy)

Applies CellTypist to automatically annotate clusters with predicted cell types.
If too few genes overlap with the reference model (as in small chr5 datasets), a robust fallback assigns pseudo-labels based on Leiden clusters, ensuring consistent output for evaluation and CI runs.

In [ ]:

import scanpy as sc, celltypist, re, pandas as pd, numpy as np
from pathlib import Path

# Use adata.raw (normalized+log1p)
if adata.raw is not None:
    adata_ct = adata.raw.to_adata()
else:
    adata_ct = adata.copy()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)

# Clean gene names to improve potential overlaps
adata_ct.var_names = [re.sub(r"[^A-Za-z0-9]", "", g).upper() for g in adata_ct.var_names]
adata_ct.var_names_make_unique()

# Load model (human by default; change to 'ImmGen' for mouse)
species = "human"
celltypist.models.download_models()
model_name = "Immune_All_High.pkl" if species=="human" else "ImmGen"
model = celltypist.models.Model.load(model=model_name)

model_feats = set([g.upper() for g in model.classifier.features])
overlap = model_feats & set([g.upper() for g in adata_ct.var_names])
print("Overlap with model features:", len(overlap))

MIN_OVERLAP = 50
used_fallback = False
if adata_ct.n_vars < MIN_OVERLAP or len(overlap) < MIN_OVERLAP:
    print("⚠️ Low overlap / tiny dataset; using pseudo-labels from Leiden clusters.")
    if "leiden" not in adata.obs:
        raise RuntimeError("Leiden missing. Run the previous cell first.")
    adata.obs["celltypist_label"] = [f"Cluster_{c}" for c in adata.obs["leiden"]]
    adata.obs["celltypist_conf"]  = 1.0
    adata.obs["celltypist_majority"] = adata.obs["leiden"].astype(str)
    used_fallback = True
else:
    pred = celltypist.annotate(adata_ct, model=model, majority_voting=True)
    adata.obs["celltypist_label"] = pred.predicted_labels
    adata.obs["celltypist_conf"]  = pred.probability.max(axis=1)
    if hasattr(pred, "majority_voting"):
        adata.obs["celltypist_majority"] = pred.majority_voting

# Save table and annotated h5ad
out = Path("week6_runtime/out/af_quant"); out.mkdir(parents=True, exist_ok=True)
adata.write(out/"adata_annotated.h5ad")
summary = (adata.obs["celltypist_label"].value_counts()
           .rename_axis("celltypist_label").reset_index(name="n_cells"))
summary.to_csv(out/"celltypist_summary.tsv", sep="\\t", index=False)
print("Annotation done.", "Fallback used." if used_fallback else "CellTypist used.")
summary.head()


## 8) Plots — final UMAPs with Leiden and CellTypist labels

In [ ]:

import scanpy as sc, matplotlib.pyplot as plt, pathlib

fig_dir = pathlib.Path("figures"); fig_dir.mkdir(parents=True, exist_ok=True)
sc.pl.umap(adata, color=["leiden","celltypist_label"], wspace=0.4, legend_loc="on data", show=False)
plt.savefig(fig_dir/"umap_final.png", dpi=220, bbox_inches="tight"); plt.close()
sc.pl.umap(adata, color="celltypist_label", legend_loc="on data", show=False)
plt.savefig(fig_dir/"umap_celltypist_only.png", dpi=220, bbox_inches="tight"); plt.close()
print("Saved figures:")
print(" - figures/umap_final.png")
print(" - figures/umap_celltypist_only.png")


---
### 🕒 Time 
- Total time: ~9 hours 

